In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import joblib

In [2]:
rf_model = joblib.load("lai_rf_model.joblib")

In [3]:
FEATURE_NAMES = [
    'mean_red',
    'mean_green',
    'mean_blue',
    'std_red',
    'std_green',
    'std_blue',
    'mean_brightness',
    'green_to_red_ratio',
    'green_to_blue_ratio'
]

def compute_rgb_features(image_path):
    rgb = np.asarray(
        Image.open(image_path).convert('RGB'),
        dtype=np.float32
    )

    red = rgb[:, :, 0]
    green = rgb[:, :, 1]
    blue = rgb[:, :, 2]

    brightness = (red + green + blue) / 3.0

    feature_values = {
        'mean_red': red.mean(),
        'mean_green': green.mean(),
        'mean_blue': blue.mean(),
        'std_red': red.std(),
        'std_green': green.std(),
        'std_blue': blue.std(),
        'mean_brightness': brightness.mean(),
        'green_to_red_ratio': green.mean() / (red.mean() + 1e-8),
        'green_to_blue_ratio': green.mean() / (blue.mean() + 1e-8)
    }

    return [feature_values[name] for name in FEATURE_NAMES]

In [4]:
def predict_lai(image_path):
    features = compute_rgb_features(image_path)

    predicted_lai = rf_model.predict([features])[0]

    return float(predicted_lai)

In [5]:
weather = pd.read_csv("weather_eto.csv")

weather.head()

,Date,Temperature,Humidity,WindSpeed,SolarRadiation,Rainfall,ET0
0,2018-01-01,13.59,37.51,1.07,9.42,0.00,1.518274
1,2018-01-02,16.65,34.38,1.06,5.10,0.00,1.799389
2,2018-01-03,16.62,31.11,1.23,4.83,0.08,2.041937
3,2018-01-04,17.57,53.09,1.25,7.88,2.71,1.673366
4,2018-01-05,16.71,70.86,1.05,9.76,1.28,1.285768


In [6]:
reference_thresholds = {
    'Indoor': {
        'low_to_moderate': 0.03198,
        'moderate_to_high': 0.05981
    },

    'Outdoor': {
        'low_to_moderate': 0.00324,
        'moderate_to_high': 0.00722
    }
}

In [7]:
def classify_lai(predicted_lai, environment):

    if environment not in reference_thresholds:
        return 'Unknown'

    thresholds = reference_thresholds[environment]

    if predicted_lai < thresholds['low_to_moderate']:
        return 'Low'

    elif predicted_lai < thresholds['moderate_to_high']:
        return 'Moderate'

    else:
        return 'High'

In [8]:
classify_lai(0.01, 'Indoor')

'Low'

In [9]:
eto_thresholds = {
    'low_to_moderate': 2.231133901283059,
    'moderate_to_high': 5.158432087433082
}

In [10]:
def classify_eto(eto):

    if eto < eto_thresholds['low_to_moderate']:
        return 'Low'

    elif eto < eto_thresholds['moderate_to_high']:
        return 'Moderate'

    else:
        return 'High'

In [11]:
rain_threshold = 2.7575

In [12]:
def adjust_for_rainfall(demand, rainfall):

    if rainfall >= rain_threshold:

        if demand == 'High':
            return 'Moderate'

        elif demand == 'Moderate':
            return 'Low'

        else:
            return 'Low'

    return demand

In [13]:
def combine_demand(lai_demand, eto_demand):

    demand_level = {
        'Low': 1,
        'Moderate': 2,
        'High': 3
    }

    final_level = max(
        demand_level[lai_demand],
        demand_level[eto_demand]
    )

    if final_level == 1:
        return 'Low'

    elif final_level == 2:
        return 'Moderate'

    else:
        return 'High'

In [14]:
def get_weather_for_date(selected_date):

    selected_date = pd.to_datetime(selected_date)

    result = weather[
        weather['Date'] == selected_date
    ]

    if result.empty:
        return None

    return result.iloc[0]

In [15]:
weather['Date'] = pd.to_datetime(weather['Date'])

In [16]:
get_weather_for_date("2021-06-15")

Date              2021-06-15 00:00:00
Temperature                      27.3
Humidity                        27.08
WindSpeed                        1.81
SolarRadiation                  30.88
Rainfall                          0.0
ET0                          7.296706
Name: 1261, dtype: object

In [17]:
def irrigation_decision(image_path, environment, selected_date):

    # 1. Predict LAI from image
    predicted_lai = predict_lai(image_path)

    # 2. Get weather for selected date
    weather_row = get_weather_for_date(selected_date)

    if weather_row is None:
        return {
            'Status': 'Invalid date',
            'Message': 'Selected date is not available.'
        }

    # 3. Get ET0 and rainfall
    eto = float(weather_row['ET0'])
    rainfall = float(weather_row['Rainfall'])

    # 4. LAI demand
    lai_demand = classify_lai(
        predicted_lai,
        environment
    )

    # 5. ET0 demand
    eto_demand = classify_eto(eto)

    # 6. Combine LAI + ET0
    combined_demand = combine_demand(
        lai_demand,
        eto_demand
    )

    # 7. Rainfall adjustment
    final_demand = adjust_for_rainfall(
        combined_demand,
        rainfall
    )

    return {
        'Date': selected_date,
        'Environment': environment,

        'LAI': predicted_lai,
        'LAI_Demand': lai_demand,

        'Temperature': float(weather_row['Temperature']),
        'Humidity': float(weather_row['Humidity']),
        'WindSpeed': float(weather_row['WindSpeed']),
        'SolarRadiation': float(weather_row['SolarRadiation']),

        'ET0': eto,
        'ET0_Demand': eto_demand,

        'Rainfall': rainfall,

        'Final_Irrigation_Demand': final_demand
    }

In [18]:
def display_result(result):

    print("======================================")
    print("          IRRIGATION DECISION")
    print("======================================")

    print(f"Date        : {result['Date']}")
    print(f"Environment : {result['Environment']}")

    print("--------------------------------------")

    print(f"LAI         : {result['LAI']:.4f}")
    print(f"LAI Demand  : {result['LAI_Demand']}")

    print("--------------------------------------")

    print(f"Temperature : {result['Temperature']:.2f} °C")
    print(f"Humidity    : {result['Humidity']:.2f} %")
    print(f"Wind Speed  : {result['WindSpeed']:.2f} m/s")
    print(f"Solar Rad.  : {result['SolarRadiation']:.2f} MJ/m²/day")

    print("--------------------------------------")

    print(f"ET₀         : {result['ET0']:.2f} mm/day")
    print(f"ET₀ Demand  : {result['ET0_Demand']}")
    print(f"Rainfall    : {result['Rainfall']:.2f} mm")

    print("--------------------------------------")

    print(
        f"FINAL DEMAND: "
        f"{result['Final_Irrigation_Demand']}"
    )

    print("======================================")

In [20]:
import tkinter as tk
from tkinter import filedialog, simpledialog, messagebox

# ------------------------------------
# USER INPUT
# ------------------------------------

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# DATE
while True:
    date_input = simpledialog.askstring(
        'Date Input',
        'Enter date (YYYY-MM-DD) between 2018-01-01 and 2023-12-31:',
        parent=root
    )
    if date_input is None:
        print('Cancelled.')
    else:
        try:
            selected_date = pd.to_datetime(date_input)
            if pd.Timestamp('2018-01-01') <= selected_date <= pd.Timestamp('2023-12-31'):
                break
            messagebox.showerror('Invalid Date', 'Please enter a date within the allowed range.')
        except:
            messagebox.showerror('Invalid Format', 'Use YYYY-MM-DD format.')

# IMAGE
while True:
    image_path = filedialog.askopenfilename(
        title='Select crop image',
        filetypes=[('Image files', '*.jpg *.jpeg *.png *.bmp *.tiff')],
        parent=root
    )
    if not image_path:
        messagebox.showerror('No Image', 'Please select a valid image file.')
    else:
        print(f'Selected image: {image_path}')
        break

# ENVIRONMENT
while True:
    environment = simpledialog.askstring(
        'Environment',
        'Enter environment (Indoor/Outdoor):',
        parent=root
    )
    if environment is None:
        print('Cancelled.')
    elif environment.strip().title() in ['Indoor', 'Outdoor']:
        environment = environment.strip().title()
        break
    else:
        messagebox.showerror('Invalid Input', 'Please enter either Indoor or Outdoor.')

root.destroy()

# ------------------------------------
# RUN ENGINE
# ------------------------------------

result = irrigation_decision(
    image_path=image_path,
    environment=environment,
    selected_date=selected_date
)

display_result(result)

Selected image: C:/Users/nihar/OneDrive/Desktop/Water To Pixels/DatasetLAI/Ground Truth Images/Indoor/IN_DAG0101.jpg
          IRRIGATION DECISION
Date        : 2018-01-01 00:00:00
Environment : Indoor
--------------------------------------
LAI         : 0.0094
LAI Demand  : Low
--------------------------------------
Temperature : 13.59 °C
Humidity    : 37.51 %
Wind Speed  : 1.07 m/s
Solar Rad.  : 9.42 MJ/m²/day
--------------------------------------
ET₀         : 1.52 mm/day
ET₀ Demand  : Low
Rainfall    : 0.00 mm
--------------------------------------
FINAL DEMAND: Low
